# 00 · PTB-XL Data Processing

Part of the study *Multi-Label 12-Lead ECG Classification on the PTB-XL Dataset* ([arXiv:2609.12803](https://arxiv.org/abs/2609.12803)). Run order, folder layout, and results: [project README](../README.md).

Raw source: **PTB-XL 1.0.3** (`data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/`).

This notebook runs on Google Colab and:

1. Unzips `ECG AI/PTB-XL.zip` into Drive `ECG AI/data/` (raw WFDB stays there).
2. Maps SCP statements to the five diagnostic superclasses (`NORM`, `MI`, `STTC`, `CD`, `HYP`).
3. Writes official `strat_fold` splits: train 1–8 / val 9 / test 10 (patient-disjoint assert).
4. Applies a **single** Butterworth band-pass 0.5–40 Hz (no 50 Hz notch; no extra baseline high-pass).
5. Computes train lead-wise mean/std and applies the same statistics to val/test.
6. Exports **two** memmap caches:
   - `processed/sr100/` — tensors `(N, 12, 1000)` for notebooks 01, 02, 02.1, 03
   - `processed/sr500/` — tensors `(N, 12, 5000)` from `filename_hr` / `records500` for ECGFounder (notebook 04)

ECGFounder trains on native 500 Hz. There is **no** forward interpolation in the training notebooks.

Set `FORCE_REBUILD=True` (or `FORCE_REBUILD_RATES`) when the source or filter settings change.


In [ ]:
!pip install -q wfdb scipy ipywidgets matplotlib pandas numpy openpyxl scikit-learn pyarrow tqdm

## Processing parameters

- `SAMPLING_RATES = (100, 500)` — export both caches. Widgets and paper figures use `SAMPLING_RATE = 100`.
- `FORCE_REBUILD=True` — rebuild every listed rate. `FORCE_REBUILD_RATES = (500,)` rebuilds only `sr500`.
- Split — official `strat_fold`: train 1–8, val 9, test 10.
- `PROCESS_OPTIONS` — default: band-pass 0.5–40 Hz, notch/baseline off; normalize = train lead-wise stats (second export pass).

Notebooks 01–03 / 02.1 read **sr100**. Notebook 04 (ECGFounder) reads **sr500**.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

# Dual cache: 100 Hz for 01–03 / 02.1, 500 Hz for ECGFounder (notebook 04).
SAMPLING_RATES = (100, 500)
SAMPLING_RATE = 100  # widgets + paper figures (does not change export rates)
FORCE_REBUILD = False  # True: rebuild every rate in SAMPLING_RATES
FORCE_REBUILD_RATES = ()  # e.g. (500,) to rebuild only processed/sr500
EXAMPLE_SEED = 42  # paper fig07 example ECG; does not change tensor splits
N_PROCESS_WORKERS = min(16, max(4, (os.cpu_count() or 8)))

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'ECG AI'
ZIP_PATH = PROJECT_DIR / 'PTB-XL.zip'
DATA_DIR = PROJECT_DIR / 'data'  # raw PTB-XL (unzip target)
PROCESSED_DIR = PROJECT_DIR / 'processed' / f'sr{SAMPLING_RATE}'
PROCESSED_DIRS = {fs: PROJECT_DIR / 'processed' / f'sr{fs}' for fs in SAMPLING_RATES}

LOCAL_ZIP = Path('/content/PTB-XL.zip')
LOCAL_WFDB_ROOT = Path('/content/ptbxl_local')  # process-only staging of records100/500
LOCAL_PROCESSED_ROOT = Path('/content/processed_local')

PROCESS_OPTIONS = {
    'bandpass': True,
    'bandpass_low': 0.5,
    'bandpass_high': 40.0,
    'notch': False,  # 50 Hz notch is off at both 100 and 500 Hz
    'notch_freq': 50.0,
    'baseline': False,  # extra high-pass off; band-pass is enough
    'baseline_cutoff': 0.5,
    'normalize': True,
    'butter_order': 4,
}

assert all(fs in (100, 500) for fs in SAMPLING_RATES), 'SAMPLING_RATES must be 100 and/or 500'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
for _p in PROCESSED_DIRS.values():
    _p.parent.mkdir(parents=True, exist_ok=True)

_data_csv = next(DATA_DIR.rglob('ptbxl_database.csv'), None) if DATA_DIR.exists() else None
_data_ready = _data_csv is not None

print('PROJECT_DIR    :', PROJECT_DIR)
print('ZIP_PATH       :', ZIP_PATH, '| exists=', ZIP_PATH.exists())
print('DATA_DIR       :', DATA_DIR, '| ready=', _data_ready, '| csv=', _data_csv)
print('SAMPLING_RATES :', SAMPLING_RATES)
print('EXPECTED RAW   :', DATA_DIR / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3')
print('FORCE_REBUILD  :', FORCE_REBUILD, '| FORCE_REBUILD_RATES=', FORCE_REBUILD_RATES)
print('N_PROCESS_WORKERS:', N_PROCESS_WORKERS)
print('SPLIT          : strat_fold train=1-8 val=9 test=10')
print('EXAMPLE_SEED   :', EXAMPLE_SEED)
_any_missing = False
for fs, pdir in PROCESSED_DIRS.items():
    ready = (pdir / 'manifest.json').exists()
    _any_missing = _any_missing or (not ready)
    print(f'processed/sr{fs} :', pdir, '| ready=', ready)
if _data_ready and _any_missing:
    print(
        'STATUS: data OK, at least one processed cache MISSING -> run Unzip/metadata '
        '(skip if csv present), Dynamic analysis, then memmap export. Do not start 01-04 yet.'
    )
elif _data_ready and not _any_missing:
    print('STATUS: data OK, sr100 and sr500 OK -> training notebooks 01-04 can run.')
elif not _data_ready:
    print('STATUS: data MISSING -> run Unzip and metadata (needs PTB-XL.zip).')


## Helper functions

Unzip, metadata, filtering, and plots. Records are never loaded fully into RAM. Export later stages `records100` / `records500` onto Colab SSD and writes memmaps with a process pool.


In [ ]:
from __future__ import annotations

import ast
import json
import os
import shutil
import subprocess
import zipfile
from concurrent.futures import ProcessPoolExecutor
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from IPython.display import clear_output, display
from scipy.signal import butter, filtfilt, iirnotch
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm.auto import tqdm
import sys

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import (
    DATASET_VERSION,
    RAW_DATASET_DIRNAME,
    assert_ptbxl_version,
    find_ptbxl_data_root,
)

SUPERCLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
LEAD_NAMES = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
TRAIN_FOLDS = (1, 2, 3, 4, 5, 6, 7, 8)  # official PTB-XL strat_fold split (train)
VAL_FOLDS = (9,)
TEST_FOLDS = (10,)


@dataclass
class ProcessOptions:
    bandpass: bool = True
    bandpass_low: float = 0.5
    bandpass_high: float = 40.0
    notch: bool = False
    notch_freq: float = 50.0
    baseline: bool = False
    baseline_cutoff: float = 0.5
    normalize: bool = True
    butter_order: int = 4

    def to_dict(self) -> dict:
        return asdict(self)


def find_data_root(start: Path) -> Path:
    root = find_ptbxl_data_root(start)
    assert_ptbxl_version(root)
    print('PTB-XL version:', DATASET_VERSION, '|', RAW_DATASET_DIRNAME)
    return root


def unzip_ptbxl(zip_path: Path, local_zip: Path, data_dir: Path) -> Path:
    """Unzip PTB-XL into Drive DATA_DIR so raw WFDB persists for AI pipelines."""
    if data_dir.exists():
        try:
            root = find_data_root(data_dir)
            print('Drive data already present:', root)
            return root
        except FileNotFoundError as exc:
            print(exc)
            print('1.0.3 missing — unzipping archive...')
    if not zip_path.exists():
        raise FileNotFoundError(
            f'Zip not found: {zip_path}\n'
            'Expected /content/drive/MyDrive/ECG AI/PTB-XL.zip'
        )
    data_dir.mkdir(parents=True, exist_ok=True)
    need_copy = (
        not local_zip.exists()
        or local_zip.stat().st_size != zip_path.stat().st_size
    )
    if need_copy:
        print('Copying zip to Colab local disk...')
        shutil.copy2(zip_path, local_zip)
    print('Unzipping to Drive data dir:', data_dir)
    with zipfile.ZipFile(local_zip, 'r') as zf:
        members = zf.infolist()
        for member in tqdm(members, desc='unzip PTB-XL', leave=True):
            zf.extract(member, data_dir)
    root = find_data_root(data_dir)
    print('DATA_ROOT:', root)
    return root


def _parse_scp_codes(value):
    if isinstance(value, dict):
        return value
    return ast.literal_eval(value)


def _load_aggregation_table(data_root: Path) -> pd.DataFrame:
    agg_df = pd.read_csv(data_root / 'scp_statements.csv', index_col=0)
    return agg_df[agg_df['diagnostic'] == 1.0].copy()


def _aggregate_superclasses(scp_codes: Dict[str, float], agg_df: pd.DataFrame) -> List[str]:
    classes = []
    for code in scp_codes.keys():
        if code in agg_df.index:
            superclass = agg_df.loc[code, 'diagnostic_class']
            if pd.notna(superclass):
                classes.append(str(superclass))
    classes = sorted(set(classes))
    return [name for name in classes if name in SUPERCLASSES]


def load_metadata(data_root: str | Path) -> pd.DataFrame:
    root = Path(data_root)
    metadata = pd.read_csv(root / 'ptbxl_database.csv', index_col='ecg_id')
    metadata['scp_codes'] = metadata['scp_codes'].apply(_parse_scp_codes)
    agg_df = _load_aggregation_table(root)
    metadata['diagnostic_superclass'] = metadata['scp_codes'].apply(
        lambda codes: _aggregate_superclasses(codes, agg_df)
    )
    metadata = metadata[metadata['diagnostic_superclass'].map(len) > 0].copy()
    return metadata


def encode_superclasses(
    labels: Sequence[Sequence[str]],
    classes: Optional[Sequence[str]] = None,
) -> Tuple[np.ndarray, MultiLabelBinarizer]:
    mlb = MultiLabelBinarizer(classes=list(classes or SUPERCLASSES))
    encoded = mlb.fit_transform(labels)
    return encoded.astype(np.float32), mlb


def _to_time_major(signal: np.ndarray) -> Tuple[np.ndarray, bool]:
    array = np.asarray(signal, dtype=np.float32)
    if array.ndim != 2:
        raise ValueError(f'ECG signal must be 2D, got shape {array.shape}')
    if array.shape[0] < array.shape[1]:
        return array.T.copy(), True
    return array.copy(), False


def _restore_layout(signal: np.ndarray, was_lead_first: bool) -> np.ndarray:
    return signal.T if was_lead_first else signal


def _safe_highcut(fs: float, high: float) -> float:
    return min(float(high), (fs / 2.0) * 0.95)


def bandpass_filter(signal, fs, low=0.5, high=40.0, order=4):
    time_major, was_lead_first = _to_time_major(signal)
    highcut = _safe_highcut(fs, high)
    lowcut = max(float(low), 0.01)
    if lowcut >= highcut:
        return _restore_layout(time_major, was_lead_first)
    nyquist = fs / 2.0
    b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype='band')
    filtered = filtfilt(b, a, time_major, axis=0)
    return _restore_layout(filtered.astype(np.float32), was_lead_first)


def notch_filter(signal, fs, freq=50.0, quality=30.0):
    time_major, was_lead_first = _to_time_major(signal)
    nyquist = fs / 2.0
    if freq >= nyquist * 0.95:
        return _restore_layout(time_major, was_lead_first)
    b, a = iirnotch(w0=freq / nyquist, Q=quality)
    filtered = filtfilt(b, a, time_major, axis=0)
    return _restore_layout(filtered.astype(np.float32), was_lead_first)


def remove_baseline_wander(signal, fs, cutoff=0.5, order=2):
    time_major, was_lead_first = _to_time_major(signal)
    nyquist = fs / 2.0
    highcut = min(max(float(cutoff), 0.01), nyquist * 0.95)
    b, a = butter(order, highcut / nyquist, btype='high')
    filtered = filtfilt(b, a, time_major, axis=0)
    return _restore_layout(filtered.astype(np.float32), was_lead_first)


def normalize_leads(signal):
    time_major, was_lead_first = _to_time_major(signal)
    mean = time_major.mean(axis=0, keepdims=True)
    std = time_major.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    normalized = (time_major - mean) / std
    return _restore_layout(normalized.astype(np.float32), was_lead_first)


def process_ecg(signal, fs, options: ProcessOptions | None = None):
    opts = options or ProcessOptions()
    processed = np.asarray(signal, dtype=np.float32)
    if opts.baseline:
        processed = remove_baseline_wander(processed, fs, cutoff=opts.baseline_cutoff)
    if opts.bandpass:
        processed = bandpass_filter(
            processed, fs, low=opts.bandpass_low, high=opts.bandpass_high, order=opts.butter_order
        )
    if opts.notch:
        processed = notch_filter(processed, fs, freq=opts.notch_freq)
    # Per-record z-score is off: train lead-wise mean/std is applied at export.
    if opts.normalize and getattr(opts, 'per_record_normalize', False):
        processed = normalize_leads(processed)
    return processed


def compute_train_lead_stats(X) -> tuple:
    """Lead-wise mean/std over TRAIN only (N, 12, T)."""
    mean = np.asarray(X).mean(axis=(0, 2))
    std = np.asarray(X).std(axis=(0, 2))
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def apply_train_lead_norm(sig, mean, std):
    return ((sig - mean[:, None]) / std[:, None]).astype(np.float32)


def to_lead_first(signal: np.ndarray) -> np.ndarray:
    array = np.asarray(signal, dtype=np.float32)
    if array.shape[0] == 12 and array.shape[0] < array.shape[1]:
        return array
    return array.T


def load_record_signal(data_root, row, sampling_rate: int = 500) -> np.ndarray:
    relative = row['filename_hr'] if sampling_rate == 500 else row['filename_lr']
    signal, _ = wfdb.rdsamp(str(Path(data_root) / relative))
    return signal.astype(np.float32)


def filter_metadata(
    metadata: pd.DataFrame,
    superclasses: Optional[Sequence[str]] = None,
    sex: Optional[int] = None,
    age_range: Optional[tuple[float, float]] = None,
    folds: Optional[Sequence[int]] = None,
    ecg_id: Optional[int] = None,
) -> pd.DataFrame:
    filtered = metadata.copy()
    if ecg_id is not None:
        if 'ecg_id' in filtered.columns:
            return filtered[filtered['ecg_id'] == int(ecg_id)].copy()
        if int(ecg_id) in filtered.index:
            return filtered.loc[[int(ecg_id)]].copy()
        return filtered.iloc[0:0].copy()
    if superclasses is not None:
        if not superclasses:
            return filtered.iloc[0:0].copy()
        wanted = set(superclasses)
        filtered = filtered[
            filtered['diagnostic_superclass'].map(lambda labels: bool(wanted.intersection(labels)))
        ]
    if sex is not None:
        filtered = filtered[filtered['sex'] == sex]
    if age_range is not None:
        low, high = age_range
        filtered = filtered[(filtered['age'] >= low) & (filtered['age'] <= high)]
    if folds is not None:
        if not folds:
            return filtered.iloc[0:0].copy()
        filtered = filtered[filtered['strat_fold'].isin(list(folds))]
    return filtered.copy()


def summarize_metadata(metadata: pd.DataFrame) -> pd.DataFrame:
    counts = {name: 0 for name in SUPERCLASSES}
    for labels in metadata['diagnostic_superclass']:
        for name in labels:
            if name in counts:
                counts[name] += 1
    return pd.DataFrame(
        {'class_name': list(counts.keys()), 'count': list(counts.values())}
    ).sort_values('class_name').reset_index(drop=True)


def plot_class_distribution(metadata, title='Diagnostic superclass distribution'):
    summary = summarize_metadata(metadata)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(summary['class_name'], summary['count'], color='#3b82f6')
    ax.set_title(title)
    ax.set_xlabel('Superclass')
    ax.set_ylabel('Record count')
    fig.tight_layout()
    return fig


def plot_fold_distribution(metadata):
    fig, ax = plt.subplots(figsize=(8, 4))
    counts = metadata['strat_fold'].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color='#0ea5e9')
    ax.set_title('Records per strat_fold')
    ax.set_xlabel('Fold')
    ax.set_ylabel('Count')
    fig.tight_layout()
    return fig


def plot_age_sex(metadata):
    fig, ax = plt.subplots(figsize=(8, 4))
    for sex_value, label in [(0, 'Male'), (1, 'Female')]:
        ages = metadata.loc[metadata['sex'] == sex_value, 'age'].dropna()
        ax.hist(ages, bins=20, alpha=0.55, label=label)
    ax.set_title('Age distribution by sex')
    ax.set_xlabel('Age')
    ax.set_ylabel('Count')
    ax.legend()
    fig.tight_layout()
    return fig


def _lead_index(lead):
    if isinstance(lead, int):
        return lead
    name = lead.upper()
    if name not in LEAD_NAMES:
        raise ValueError(f'Unknown lead {lead}. Expected one of {LEAD_NAMES}')
    return LEAD_NAMES.index(name)


def plot_12_lead_ecg(signal, fs, title='12-lead ECG', leads=None):
    array = np.asarray(signal)
    if array.shape[0] < array.shape[1]:
        array = array.T
    selected = list(leads) if leads is not None else LEAD_NAMES
    n_leads = len(selected)
    time = np.arange(array.shape[0]) / float(fs)
    fig, axes = plt.subplots(n_leads, 1, figsize=(14, 1.35 * n_leads), sharex=True)
    if n_leads == 1:
        axes = [axes]
    for ax, lead_name in zip(axes, selected):
        idx = _lead_index(lead_name)
        ax.plot(time, array[:, idx], color='#111827', linewidth=0.8)
        ax.set_ylabel(lead_name, rotation=0, labelpad=18, va='center')
        ax.grid(True, alpha=0.25)
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    return fig


def plot_raw_vs_processed(raw, processed, fs, lead='II', title=None):
    raw_arr = np.asarray(raw)
    proc_arr = np.asarray(processed)
    if raw_arr.shape[0] < raw_arr.shape[1]:
        raw_arr = raw_arr.T
    if proc_arr.shape[0] < proc_arr.shape[1]:
        proc_arr = proc_arr.T
    idx = _lead_index(lead)
    time = np.arange(raw_arr.shape[0]) / float(fs)
    lead_name = LEAD_NAMES[idx] if isinstance(lead, int) else lead.upper()
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(time, raw_arr[:, idx], color='#6b7280', linewidth=0.9)
    axes[0].set_title(f'Raw lead {lead_name}')
    axes[0].grid(True, alpha=0.25)
    axes[1].plot(time, proc_arr[:, idx], color='#2563eb', linewidth=0.9)
    axes[1].set_title(f'Processed lead {lead_name}')
    axes[1].set_xlabel('Time (s)')
    axes[1].grid(True, alpha=0.25)
    if title:
        fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig


def expected_length(fs: int) -> int:
    return 1000 if int(fs) == 100 else 5000


def pad_or_crop(sig: np.ndarray, length: int) -> np.ndarray:
    if sig.shape[1] == length:
        return sig
    if sig.shape[1] > length:
        return sig[:, :length]
    out = np.zeros((12, length), dtype=np.float32)
    out[:, : sig.shape[1]] = sig
    return out


def assign_official_fold_splits(
    metadata: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if 'strat_fold' not in metadata.columns:
        raise KeyError('metadata missing strat_fold')
    if metadata['strat_fold'].isna().any():
        raise ValueError('strat_fold contains NA')
    folds = metadata['strat_fold'].astype(int)
    allowed = set(TRAIN_FOLDS) | set(VAL_FOLDS) | set(TEST_FOLDS)
    unexpected = sorted(set(folds.unique()) - allowed)
    if unexpected:
        raise ValueError(f'Unexpected strat_fold values: {unexpected}')

    train_df = metadata[folds.isin(TRAIN_FOLDS)].copy()
    val_df = metadata[folds.isin(VAL_FOLDS)].copy()
    test_df = metadata[folds.isin(TEST_FOLDS)].copy()
    if len(train_df) + len(val_df) + len(test_df) != len(metadata):
        raise RuntimeError('strat_fold split does not cover all records')

    train_p = set(train_df['patient_id'])
    val_p = set(val_df['patient_id'])
    test_p = set(test_df['patient_id'])
    if not train_p.isdisjoint(val_p) or not train_p.isdisjoint(test_p) or not val_p.isdisjoint(test_p):
        raise RuntimeError('Patient leakage across strat_fold splits')
    return train_df, val_df, test_df




def apply_train_lead_norm_block(block, mean, std):
    """Vectorized lead-wise norm for a memmap chunk (N, 12, T)."""
    return ((block - mean[None, :, None]) / std[None, :, None]).astype(np.float32)


def apply_lead_norm_memmap(path: Path, mean, std, chunk: int = 512):
    X = np.load(path, mmap_mode='r+')
    n = int(X.shape[0])
    for start in tqdm(range(0, n, chunk), desc=f'lead-norm {path.name}', leave=True):
        stop = min(start + chunk, n)
        X[start:stop] = apply_train_lead_norm_block(X[start:stop], mean, std)
    X.flush()
    del X


def _count_dat(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(1 for _ in folder.rglob('*.dat'))


def stage_records(drive_data_root: Path, local_root: Path, fs: int) -> Path:
    """Copy records100 or records500 Drive -> Colab SSD once. Training does not read this path."""
    folder = 'records500' if int(fs) == 500 else 'records100'
    src = drive_data_root / folder
    dst = local_root / folder
    if not src.is_dir():
        raise FileNotFoundError(f'{folder} missing under {drive_data_root}')
    n_src = _count_dat(src)
    n_dst = _count_dat(dst)
    if n_src <= 0:
        raise FileNotFoundError(f'no .dat files in {src}')
    if n_dst >= n_src:
        print(f'local {folder} already staged ({n_dst} .dat): {dst}')
        return local_root
    print(f'Copying {folder} -> {dst}  ({n_src} .dat, sequential). This is process-only.')
    local_root.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    subprocess.run(['cp', '-a', str(src), str(local_root)], check=True)
    print(f'staged {folder}:', dst, '| .dat', _count_dat(dst))
    return local_root


def _process_one_record(job):
    data_root, relative, fs, opt_dict, length = job
    signal, _ = wfdb.rdsamp(str(Path(data_root) / relative))
    processed = process_ecg(signal.astype(np.float32), fs=fs, options=ProcessOptions(**opt_dict))
    return pad_or_crop(to_lead_first(processed), int(length))


def processed_cache_ok(processed_dir: Path, fs: int) -> bool:
    """True when Drive cache matches official folds and native length for `fs`."""
    try:
        from ecg_common import load_processed_manifest
        manifest = load_processed_manifest(processed_dir)
    except (FileNotFoundError, ValueError, json.JSONDecodeError):
        return False
    x_train = processed_dir / 'X_train.npy'
    if not x_train.exists():
        return False
    expected = expected_length(fs)
    shape = tuple(np.load(x_train, mmap_mode='r').shape[1:])
    return shape == (12, expected) and int(manifest.get('n_samples', 0)) == expected and int(manifest.get('sampling_rate', 0)) == int(fs)


print('helpers ready')


## Unzip and metadata


In [ ]:
DATA_ROOT = unzip_ptbxl(ZIP_PATH, LOCAL_ZIP, DATA_DIR)
metadata = load_metadata(DATA_ROOT)
print('records after empty-label drop:', len(metadata))
print('patients:', metadata['patient_id'].nunique())
print('folds:', sorted(metadata['strat_fold'].unique().tolist()))
print('classes:', SUPERCLASSES)
display(summarize_metadata(metadata))
display(metadata.reset_index()[['ecg_id', 'patient_id', 'age', 'sex', 'strat_fold', 'diagnostic_superclass', 'filename_hr']].head())


## Resume / new Colab session

If `data/` is already on Drive, the zip is not extracted again. In a fresh session run, in order:

1. **pip + path + helpers** — every session
2. **Unzip and metadata** — skips automatically when the CSV is present (`Drive data already present`); still run it so `DATA_ROOT` / `metadata` exist
3. **Dynamic analysis** — splits + labels (required)
4. **Filter / widget panel** — optional preview
5. **Memmap export** — writes `processed/sr100/` and `processed/sr500/` (required before training)
6. **Verify** — check both manifests

`experiments/` stays empty here; training notebooks 01–04 create it. After a successful export set `FORCE_REBUILD = False` in the path cell.


## Dynamic analysis

Official fold counts, superclass prevalence, and age/sex. This cell also builds the train/val/test frames and multi-hot labels used by the export cell.


In [ ]:
fig_class = plot_class_distribution(metadata)
fig_fold = plot_fold_distribution(metadata)
fig_age = plot_age_sex(metadata)
display(fig_class)
display(fig_fold)
display(fig_age)

train_df, val_df, test_df = assign_official_fold_splits(metadata)
y_train, mlb = encode_superclasses(train_df['diagnostic_superclass'].tolist())
y_val = mlb.transform(val_df['diagnostic_superclass'].tolist()).astype(np.float32)
y_test = mlb.transform(test_df['diagnostic_superclass'].tolist()).astype(np.float32)

split_summary = {
    'sampling_rate': SAMPLING_RATE,
    'classes': list(mlb.classes_),
    'split_by': 'strat_fold',
    'train_folds': list(TRAIN_FOLDS),
    'val_folds': list(VAL_FOLDS),
    'test_folds': list(TEST_FOLDS),
    'n_train': int(len(train_df)),
    'n_val': int(len(val_df)),
    'n_test': int(len(test_df)),
    'n_total': int(len(metadata)),
    'n_train_patients': int(train_df['patient_id'].nunique()),
    'n_val_patients': int(val_df['patient_id'].nunique()),
    'n_test_patients': int(test_df['patient_id'].nunique()),
    'class_prevalence_train': {
        name: float(y_train[:, i].mean()) for i, name in enumerate(mlb.classes_)
    },
}
print(json.dumps(split_summary, indent=2))
plt.close('all')

## Filter and process panel

Left-side filters narrow the metadata table. Process checkboxes preview filter steps on one record.

Export uses `PROCESS_OPTIONS` and `SAMPLING_RATES` from the path/config cell. Widgets (including the Hz dropdown) are **preview only**. To copy widget process values into the export config, run the next cell.

**Summary:** the path/config cell is the source of truth; this panel does not write training tensors.


In [ ]:
import ipywidgets as widgets

age_valid = metadata['age'].dropna()
age_min = int(age_valid.min()) if len(age_valid) else 0
age_max = int(age_valid.max()) if len(age_valid) else 100

superclass_w = widgets.SelectMultiple(
    options=SUPERCLASSES,
    value=tuple(SUPERCLASSES),
    description='Class',
    layout=widgets.Layout(width='220px', height='120px'),
)
sex_w = widgets.Dropdown(
    options=[('All', None), ('Male (0)', 0), ('Female (1)', 1)],
    value=None,
    description='Sex',
)
age_w = widgets.IntRangeSlider(
    value=(age_min, age_max),
    min=age_min,
    max=age_max,
    step=1,
    description='Age',
)
fold_w = widgets.SelectMultiple(
    options=list(range(1, 11)),
    value=tuple(range(1, 11)),
    description='Fold',
    layout=widgets.Layout(width='220px', height='140px'),
)
fs_w = widgets.RadioButtons(
    options=[('100 Hz', 100), ('500 Hz', 500)],
    value=int(SAMPLING_RATE),
    description='FS',
)
ecg_id_w = widgets.IntText(value=0, description='ECG ID')
record_w = widgets.Dropdown(options=[], description='Pick ID')
lead_w = widgets.Dropdown(options=LEAD_NAMES, value='II', description='Lead')

bandpass_w = widgets.Checkbox(value=PROCESS_OPTIONS['bandpass'], description='Bandpass')
low_w = widgets.FloatSlider(value=PROCESS_OPTIONS['bandpass_low'], min=0.1, max=5.0, step=0.1, description='Low Hz')
high_w = widgets.FloatSlider(value=PROCESS_OPTIONS['bandpass_high'], min=10.0, max=100.0, step=1.0, description='High Hz')
notch_w = widgets.Checkbox(value=PROCESS_OPTIONS['notch'], description='Notch 50 Hz')
baseline_w = widgets.Checkbox(value=PROCESS_OPTIONS['baseline'], description='Baseline')
norm_w = widgets.Checkbox(value=PROCESS_OPTIONS['normalize'], description='Normalize')

status_w = widgets.HTML(value='')
table_out = widgets.Output()
plot_out = widgets.Output()
MAX_DROPDOWN = 300


def current_filtered() -> pd.DataFrame:
    ecg_id = int(ecg_id_w.value) if int(ecg_id_w.value) > 0 else None
    return filter_metadata(
        metadata,
        superclasses=list(superclass_w.value),
        sex=sex_w.value,
        age_range=tuple(age_w.value),
        folds=list(fold_w.value),
        ecg_id=ecg_id,
    )


def refresh_record_options(*_args):
    filtered = current_filtered()
    ids = list(filtered.index.astype(int))
    record_w.options = ids[:MAX_DROPDOWN]
    status_w.value = (
        f'<b>{len(filtered)}</b> records matched. Dropdown shows the first {min(len(ids), MAX_DROPDOWN)}.'
    )
    with table_out:
        clear_output(wait=True)
        preview = filtered.reset_index()[
            ['ecg_id', 'age', 'sex', 'strat_fold', 'diagnostic_superclass']
        ].head(12)
        display(preview)


def render_ecg(*_args):
    filtered = current_filtered()
    with plot_out:
        clear_output(wait=True)
        if filtered.empty:
            print('No records match the filter.')
            return
        chosen = record_w.value if record_w.value in filtered.index else filtered.index[0]
        row = filtered.loc[chosen]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        fs = int(fs_w.value)
        raw = load_record_signal(DATA_ROOT, row, sampling_rate=fs)
        options = ProcessOptions(
            bandpass=bool(bandpass_w.value),
            bandpass_low=float(low_w.value),
            bandpass_high=float(high_w.value),
            notch=bool(notch_w.value),
            baseline=bool(baseline_w.value),
            normalize=bool(norm_w.value),
        )
        processed = process_ecg(raw, fs=fs, options=options)
        labels = ', '.join(row['diagnostic_superclass'])
        title = f'ECG {int(chosen)} | {labels} | fold {int(row["strat_fold"])} | {fs} Hz'
        fig12 = plot_12_lead_ecg(processed, fs=fs, title=title)
        figcmp = plot_raw_vs_processed(raw, processed, fs=fs, lead=lead_w.value, title=title)
        display(fig12)
        display(figcmp)
        plt.close(fig12)
        plt.close(figcmp)


for widget in [superclass_w, sex_w, age_w, fold_w, ecg_id_w]:
    widget.observe(refresh_record_options, names='value')
for widget in [record_w, fs_w, lead_w, bandpass_w, low_w, high_w, notch_w, baseline_w, norm_w]:
    widget.observe(render_ecg, names='value')

filters_box = widgets.VBox([
    widgets.HTML('<b>Filter</b>'),
    superclass_w, sex_w, age_w, fold_w, fs_w, ecg_id_w, record_w, status_w,
])
process_box = widgets.VBox([
    widgets.HTML('<b>Process (preview)</b>'),
    bandpass_w, low_w, high_w, notch_w, baseline_w, norm_w, lead_w,
])
display(widgets.HBox([filters_box, process_box]))
display(table_out, plot_out)
refresh_record_options()
render_ecg()


In [ ]:
# Optional: copy widget process settings into the export config
USE_WIDGET_PROCESS_FOR_EXPORT = False

if USE_WIDGET_PROCESS_FOR_EXPORT:
    PROCESS_OPTIONS.update({
        'bandpass': bool(bandpass_w.value),
        'bandpass_low': float(low_w.value),
        'bandpass_high': float(high_w.value),
        'notch': bool(notch_w.value),
        'baseline': bool(baseline_w.value),
        'normalize': bool(norm_w.value),
    })
print('Export process options:')
print(json.dumps(PROCESS_OPTIONS, indent=2))


## AI-ready memmap export

Raw WFDB is read from Drive `ECG AI/data/`. For speed, `records100` / `records500` are copied once to Colab SSD (`/content/ptbxl_local`), then CPU workers write memmaps on local disk and copy them to Drive `processed/sr{FS}/`.

Same labels, same official folds, same band-pass for both rates:

- **sr100** — `filename_lr`, length 1000 — notebooks 01, 02, 02.1, 03
- **sr500** — `filename_hr`, length 5000 — notebook 04 (ECGFounder, no interpolate)

Skip a rate when its `manifest.json` is valid and `FORCE_REBUILD` is false.

**Before this cell:** path, helpers, Unzip and metadata, Dynamic analysis (widgets optional). Training reads the Drive caches, not `/content`.


In [ ]:
def export_split(frame, targets, split_name, data_root, fs, options, out_dir, n_workers: int):
    frame = frame.reset_index()
    n = len(frame)
    t = expected_length(fs)
    x_path = out_dir / f'X_{split_name}.npy'
    X = np.lib.format.open_memmap(str(x_path), mode='w+', dtype=np.float32, shape=(n, 12, t))
    ids = frame['ecg_id'].to_numpy(dtype=np.int64)
    patient_ids = frame['patient_id'].to_numpy(dtype=np.int64)
    col = 'filename_hr' if int(fs) == 500 else 'filename_lr'
    opt_dict = {**options.to_dict(), 'normalize': False}
    jobs = [(str(data_root), str(rel), int(fs), opt_dict, int(t)) for rel in frame[col].tolist()]
    workers = max(1, int(n_workers))
    print(f'export {split_name} @ {fs} Hz: n={n} workers={workers} from {data_root}')
    with ProcessPoolExecutor(max_workers=workers) as pool:
        mapped = pool.map(_process_one_record, jobs, chunksize=32)
        for i, sig in enumerate(tqdm(mapped, total=n, desc=f'export {split_name} {fs}Hz', leave=True)):
            X[i] = sig
    X.flush()
    del X
    np.save(out_dir / f'y_{split_name}.npy', targets.astype(np.float32))
    np.save(out_dir / f'ids_{split_name}.npy', ids)
    np.save(out_dir / f'patient_ids_{split_name}.npy', patient_ids)
    return {
        'X': f'X_{split_name}.npy',
        'y': f'y_{split_name}.npy',
        'ids': f'ids_{split_name}.npy',
        'patient_ids': f'patient_ids_{split_name}.npy',
        'n': int(n),
        'shape': [int(n), 12, int(t)],
    }


def copy_processed_to_drive(local_dir: Path, drive_dir: Path):
    drive_dir.mkdir(parents=True, exist_ok=True)
    items = list(local_dir.iterdir())
    for src in tqdm(items, desc='copy to Drive', leave=True):
        dst = drive_dir / src.name
        print(f'Copy {src.name} -> {dst}')
        if src.is_dir():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)


def export_rate(fs: int) -> None:
    drive_dir = PROCESSED_DIRS[int(fs)]
    rebuild = bool(FORCE_REBUILD) or int(fs) in {int(x) for x in FORCE_REBUILD_RATES}
    if processed_cache_ok(drive_dir, fs) and not rebuild:
        print(f'Existing processed/sr{fs} found, skip export.')
        print('Drive cache:', drive_dir)
        return

    local_dir = LOCAL_PROCESSED_ROOT / f'sr{int(fs)}'
    wfdb_root = stage_records(DATA_ROOT, LOCAL_WFDB_ROOT, fs)
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    if int(fs) == int(SAMPLING_RATE):
        figures_dir = local_dir / 'figures'
        figures_dir.mkdir(parents=True, exist_ok=True)
        fig_class = plot_class_distribution(metadata)
        fig_class.savefig(figures_dir / 'class_distribution.png', dpi=150)
        plt.close(fig_class)
        fig_fold = plot_fold_distribution(metadata)
        fig_fold.savefig(figures_dir / 'fold_distribution.png', dpi=150)
        plt.close(fig_fold)
        fig_age = plot_age_sex(metadata)
        fig_age.savefig(figures_dir / 'age_sex.png', dpi=150)
        plt.close(fig_age)

    options = ProcessOptions(**PROCESS_OPTIONS)
    splits_meta = {}
    for split_name, frame, targets in [
        ('train', train_df, y_train),
        ('val', val_df, y_val),
        ('test', test_df, y_test),
    ]:
        splits_meta[split_name] = export_split(
            frame, targets, split_name, wfdb_root, fs, options, local_dir,
            n_workers=N_PROCESS_WORKERS,
        )

    X_train = np.load(local_dir / 'X_train.npy', mmap_mode='r')
    lead_mean, lead_std = compute_train_lead_stats(X_train)
    del X_train
    norm_stats = {
        'mode': 'train_leadwise',
        'leads': ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6'],
        'mean': lead_mean.tolist(),
        'std': lead_std.tolist(),
    }
    (local_dir / 'norm_stats.json').write_text(json.dumps(norm_stats, indent=2), encoding='utf-8')
    if options.normalize:
        for split_name in ('train', 'val', 'test'):
            apply_lead_norm_memmap(local_dir / f'X_{split_name}.npy', lead_mean, lead_std)

    meta_export = metadata.reset_index().copy()
    meta_export['diagnostic_superclass'] = meta_export['diagnostic_superclass'].map(
        lambda labels: json.dumps(list(labels))
    )
    meta_export['scp_codes'] = meta_export['scp_codes'].map(lambda codes: json.dumps(codes))
    meta_export.to_parquet(local_dir / 'metadata.parquet', index=False)

    n_samples = expected_length(fs)
    source = 'filename_hr_records500' if int(fs) == 500 else 'filename_lr_records100'
    rate_summary = dict(split_summary)
    rate_summary['sampling_rate'] = int(fs)
    (local_dir / 'split_summary.json').write_text(json.dumps(rate_summary, indent=2), encoding='utf-8')
    (local_dir / 'process_options.json').write_text(json.dumps(options.to_dict(), indent=2), encoding='utf-8')

    manifest = {
        'created_at': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'source_zip': str(ZIP_PATH),
        'raw_root_name': DATA_ROOT.name,
        'dataset_version': next((p for p in DATA_ROOT.name.split('-') if p[:1].isdigit()), DATA_ROOT.name),
        'sampling_rate': int(fs),
        'n_leads': 12,
        'n_samples': int(n_samples),
        'layout': 'lead_first',
        'dtype': 'float32',
        'classes': list(mlb.classes_),
        'split_by': 'strat_fold',
        'train_folds': list(TRAIN_FOLDS),
        'val_folds': list(VAL_FOLDS),
        'test_folds': list(TEST_FOLDS),
        'process_options': options.to_dict(),
        'norm_stats': 'norm_stats.json',
        'patient_disjoint': True,
        'source': source,
        'notebook': '00_ptbxl_colab_process',
        'process_workers': int(N_PROCESS_WORKERS),
        'splits': splits_meta,
        'files': {
            'metadata': 'metadata.parquet',
            'split_summary': 'split_summary.json',
            'process_options': 'process_options.json',
            'split_index': 'split_index.parquet',
            'norm_stats': 'norm_stats.json',
        },
    }
    split_index = pd.concat(
        [
            train_df.reset_index()[['ecg_id', 'patient_id', 'filename_lr', 'filename_hr', 'strat_fold']].assign(split='train'),
            val_df.reset_index()[['ecg_id', 'patient_id', 'filename_lr', 'filename_hr', 'strat_fold']].assign(split='val'),
            test_df.reset_index()[['ecg_id', 'patient_id', 'filename_lr', 'filename_hr', 'strat_fold']].assign(split='test'),
        ],
        ignore_index=True,
    )
    split_index.to_parquet(local_dir / 'split_index.parquet', index=False)
    (local_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(f'Copying processed/sr{fs} to Drive (training cache)...')
    copy_processed_to_drive(local_dir, drive_dir)
    print('Done:', drive_dir / 'manifest.json')


_missing_export_deps = [
    name for name in (
        'DATA_ROOT', 'metadata', 'train_df', 'val_df', 'test_df',
        'y_train', 'y_val', 'y_test', 'mlb', 'split_summary',
    )
    if name not in globals()
]
if _missing_export_deps:
    raise RuntimeError(
        'Export prerequisites missing in this session: '
        + ', '.join(_missing_export_deps)
        + '\nRun path setup, helpers, Unzip and metadata, then Dynamic analysis before this cell.'
    )

for _fs in SAMPLING_RATES:
    export_rate(int(_fs))


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import load_processed_manifest

for fs, pdir in PROCESSED_DIRS.items():
    manifest = load_processed_manifest(pdir)
    X_train = np.load(pdir / 'X_train.npy', mmap_mode='r')
    y_train_loaded = np.load(pdir / 'y_train.npy', mmap_mode='r')
    ids_train = np.load(pdir / 'ids_train.npy', mmap_mode='r')
    print('---')
    print(f'sr{fs}', pdir)
    print('sampling_rate', manifest.get('sampling_rate'), '| n_samples', manifest.get('n_samples'))
    print('source', manifest.get('source'))
    print('X_train', X_train.shape, X_train.dtype, '| y', y_train_loaded.shape, '| ids', ids_train[:5])
    assert tuple(X_train.shape[1:]) == (12, expected_length(fs)), X_train.shape
print('processed ready: 01-03 / 02.1 -> sr100; 04 ECGFounder -> sr500')


## Paper figures: dataset introduction

Publication figures for the Methods / Data section.

**Prerequisite:** run the path cell and the helpers cell. This cell reloads metadata from Drive `data/` if `metadata` is not already in the session. Raw zip is not required.

**Outputs** (PDF + 300 dpi PNG, no in-figure titles):

`/content/drive/MyDrive/ECG AI/paper/dataset_figures/`

Also writes `dataset_stats.json` and `dataset_stats.csv` for tables. Record universe matches `load_metadata`: diagnostic-superclass-empty ECGs are excluded.

In [ ]:
# Paper-quality PTB-XL dataset figures (Methods / Data).
# Relies on helpers already defined in this notebook: PROJECT_DIR, DATA_DIR,
# SUPERCLASSES, LEAD_NAMES, find_data_root, load_metadata, encode_superclasses,
# load_record_signal, SAMPLING_RATE.

from collections import Counter
from IPython.display import Markdown, display

PAPER_FIG_DIR = PROJECT_DIR / 'paper' / 'dataset_figures'
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)

MM_IN = 1.0 / 25.4
COL_SINGLE = 89 * MM_IN
COL_DOUBLE = 183 * MM_IN

OKABE_ITO = ['#0072B2', '#D55E00', '#E69F00', '#009E73', '#CC79A7']
CLASS_COLOR = {name: OKABE_ITO[i] for i, name in enumerate(SUPERCLASSES)}
LEAD_GRID = [
    ['I', 'AVR', 'V1', 'V4'],
    ['II', 'AVL', 'V2', 'V5'],
    ['III', 'AVF', 'V3', 'V6'],
]
NOISE_COLS = ['baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems']


def apply_paper_style() -> None:
    plt.rcParams.update({
        'font.family': 'DejaVu Sans',
        'font.size': 8,
        'axes.labelsize': 8,
        'axes.titlesize': 8,
        'xtick.labelsize': 7,
        'ytick.labelsize': 7,
        'legend.fontsize': 7,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.linewidth': 0.6,
        'xtick.major.width': 0.6,
        'ytick.major.width': 0.6,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.03,
        'figure.dpi': 120,
    })


def save_paper_fig(fig, stem: str) -> None:
    pdf_path = PAPER_FIG_DIR / f'{stem}.pdf'
    png_path = PAPER_FIG_DIR / f'{stem}.png'
    fig.savefig(pdf_path)
    fig.savefig(png_path, dpi=300)
    display(fig)
    plt.close(fig)
    print(f'saved {pdf_path.name}  {png_path.name}')


def _ensure_ecg_id_column(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if 'ecg_id' not in out.columns:
        out = out.reset_index()
        if 'ecg_id' not in out.columns:
            out = out.rename(columns={out.columns[0]: 'ecg_id'})
    return out


def _combo_key(labels) -> str:
    ordered = [name for name in SUPERCLASSES if name in set(labels)]
    return '+'.join(ordered) if ordered else '(empty)'


def _record_is_clean(row: pd.Series) -> bool:
    for col in NOISE_COLS:
        if col not in row.index:
            continue
        value = row[col]
        if pd.notna(value) and str(value).strip() != '':
            return False
    return True


def _y_matrix(frame: pd.DataFrame) -> np.ndarray:
    encoded, _ = encode_superclasses(frame['diagnostic_superclass'].tolist(), SUPERCLASSES)
    return encoded.astype(np.int32)


def _prevalence_counts(y: np.ndarray) -> dict:
    n = int(y.shape[0])
    counts = y.sum(axis=0).astype(int)
    return {
        name: {'n': int(counts[i]), 'pct': float(100.0 * counts[i] / n) if n else 0.0}
        for i, name in enumerate(SUPERCLASSES)
    }


def _load_subclass_table(data_root: Path) -> pd.DataFrame:
    agg = pd.read_csv(data_root / 'scp_statements.csv', index_col=0)
    agg = agg[agg['diagnostic'] == 1.0].copy()
    return agg


def _subclass_labels(scp_codes: dict, agg: pd.DataFrame):
    found = []
    for code in scp_codes.keys():
        if code not in agg.index:
            continue
        sub = agg.loc[code, 'diagnostic_subclass']
        if pd.notna(sub):
            found.append(str(sub))
    return sorted(set(found))


def plot_superclass_prevalence(frame: pd.DataFrame, y: np.ndarray):
    n = int(y.shape[0])
    counts = y.sum(axis=0).astype(int)
    fig, ax = plt.subplots(figsize=(COL_SINGLE, 2.4))
    ypos = np.arange(len(SUPERCLASSES))
    colors = [CLASS_COLOR[name] for name in SUPERCLASSES]
    ax.barh(ypos, counts, color=colors, height=0.72, edgecolor='none')
    ax.set_yticks(ypos)
    ax.set_yticklabels(SUPERCLASSES)
    ax.invert_yaxis()
    ax.set_xlabel('Records (multi-label)')
    xmax = max(int(counts.max()) * 1.22, 1)
    ax.set_xlim(0, xmax)
    for i, count in enumerate(counts):
        pct = 100.0 * count / n
        ax.text(count + xmax * 0.01, i, f'{int(count):,}  ({pct:.1f}%)', va='center', fontsize=7)
    fig.tight_layout()
    return fig


def plot_cooccurrence(y: np.ndarray):
    counts = y.T @ y
    jaccard = np.zeros((5, 5), dtype=float)
    for i in range(5):
        for j in range(5):
            union = counts[i, i] + counts[j, j] - counts[i, j]
            jaccard[i, j] = float(counts[i, j] / union) if union else 0.0
    fig, ax = plt.subplots(figsize=(COL_SINGLE, COL_SINGLE))
    cmap = plt.cm.YlGnBu
    im = ax.imshow(jaccard, cmap=cmap, vmin=0.0, vmax=1.0)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.set_xticklabels(SUPERCLASSES)
    ax.set_yticklabels(SUPERCLASSES)
    ax.tick_params(length=0)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    for i in range(5):
        for j in range(5):
            value = jaccard[i, j]
            text_color = 'white' if value > 0.55 else 'black'
            ax.text(
                j, i, f'{int(counts[i, j]):,}\n{value:.2f}',
                ha='center', va='center', fontsize=6.5, color=text_color,
            )
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Jaccard index')
    cbar.ax.tick_params(labelsize=7, length=2, width=0.6)
    ax.set_xlabel('Class j')
    ax.set_ylabel('Class i')
    fig.tight_layout()
    return fig, counts, jaccard


def plot_cardinality_and_combos(frame: pd.DataFrame):
    card = frame['diagnostic_superclass'].map(len)
    card_counts = card.value_counts().sort_index()
    combo_counts = Counter(_combo_key(labels) for labels in frame['diagnostic_superclass'])
    top = combo_counts.most_common(10)
    fig, axes = plt.subplots(1, 2, figsize=(COL_DOUBLE, 2.8), gridspec_kw={'width_ratios': [1, 1.45]})
    axes[0].bar(
        [str(k) for k in card_counts.index],
        card_counts.values,
        color='#0072B2',
        width=0.7,
        edgecolor='none',
    )
    axes[0].set_xlabel('Labels per record')
    axes[0].set_ylabel('Records')
    for x, v in zip(card_counts.index, card_counts.values):
        axes[0].text(str(x), v, f'{int(v):,}', ha='center', va='bottom', fontsize=6.5)
    names = [item[0] for item in top][::-1]
    vals = [item[1] for item in top][::-1]
    axes[1].barh(np.arange(len(names)), vals, color='#56B4E9', height=0.72, edgecolor='none')
    axes[1].set_yticks(np.arange(len(names)))
    axes[1].set_yticklabels(names)
    axes[1].set_xlabel('Records')
    xmax = max(vals) * 1.18 if vals else 1
    axes[1].set_xlim(0, xmax)
    for i, v in enumerate(vals):
        axes[1].text(v + xmax * 0.01, i, f'{int(v):,}', va='center', fontsize=6.5)
    fig.tight_layout()
    return fig, dict(card_counts), top


def plot_age_sex(frame: pd.DataFrame):
    bins = np.arange(0, 101, 5)
    centers = bins[:-1] + 2.5
    male = frame.loc[frame['sex'] == 0, 'age'].dropna().to_numpy()
    female = frame.loc[frame['sex'] == 1, 'age'].dropna().to_numpy()
    h_m, _ = np.histogram(male, bins=bins)
    h_f, _ = np.histogram(female, bins=bins)
    fig, ax = plt.subplots(figsize=(COL_SINGLE, 3.1))
    ax.barh(centers, -h_m, height=4.2, color='#0072B2', label='Male', edgecolor='none')
    ax.barh(centers, h_f, height=4.2, color='#CC79A7', label='Female', edgecolor='none')
    ax.set_xlabel('Records')
    ax.set_ylabel('Age (years)')
    ax.set_ylim(0, 100)
    xmax = int(max(h_m.max() if len(h_m) else 0, h_f.max() if len(h_f) else 0) * 1.15) or 1
    ax.set_xlim(-xmax, xmax)
    ax.xaxis.set_major_formatter(lambda x, _pos: f'{int(abs(x)):,}')
    ax.legend(frameon=False, loc='lower right')
    n_m, n_f = int(len(male)), int(len(female))
    ax.text(0.02, 0.98, f'M {n_m:,}', transform=ax.transAxes, va='top', color='#0072B2', fontsize=7)
    ax.text(0.98, 0.98, f'F {n_f:,}', transform=ax.transAxes, va='top', ha='right', color='#CC79A7', fontsize=7)
    fig.tight_layout()
    return fig


def plot_subclass_counts(frame: pd.DataFrame, agg: pd.DataFrame):
    subs = frame['scp_codes'].map(lambda codes: _subclass_labels(codes, agg))
    counts = Counter(name for row in subs for name in row)
    super_of = {}
    for code, row in agg.iterrows():
        sub = row['diagnostic_subclass']
        cls = row['diagnostic_class']
        if pd.notna(sub) and pd.notna(cls) and str(sub) not in super_of:
            super_of[str(sub)] = str(cls)
    items = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    names = [k for k, _ in items]
    vals = [v for _, v in items]
    colors = [CLASS_COLOR.get(super_of.get(name, ''), '#999999') for name in names]
    fig, ax = plt.subplots(figsize=(COL_DOUBLE, 5.6))
    ypos = np.arange(len(names))
    ax.barh(ypos, vals, color=colors, height=0.75, edgecolor='none')
    ax.set_yticks(ypos)
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel('Records (multi-label)')
    xmax = max(vals) * 1.16 if vals else 1
    ax.set_xlim(0, xmax)
    for i, v in enumerate(vals):
        ax.text(v + xmax * 0.008, i, f'{int(v):,}', va='center', fontsize=6)
    handles = [plt.Rectangle((0, 0), 1, 1, color=CLASS_COLOR[c]) for c in SUPERCLASSES]
    ax.legend(handles, SUPERCLASSES, frameon=False, ncol=5, loc='lower right', bbox_to_anchor=(1, 1.01))
    fig.tight_layout()
    return fig, dict(counts), super_of


def plot_split_balance(split_frames, source: str):
    order = [name for name in ('train', 'val', 'test') if name in split_frames]
    x = np.arange(len(SUPERCLASSES))
    width = 0.24
    fig, ax = plt.subplots(figsize=(COL_DOUBLE, 2.7))
    palette = {'train': '#0072B2', 'val': '#E69F00', 'test': '#D55E00'}
    for i, split_name in enumerate(order):
        y = _y_matrix(split_frames[split_name])
        n = max(int(y.shape[0]), 1)
        pct = 100.0 * y.sum(axis=0) / n
        ax.bar(
            x + (i - (len(order) - 1) / 2) * width,
            pct,
            width=width,
            color=palette[split_name],
            label=f'{split_name} (n={y.shape[0]:,})',
            edgecolor='none',
        )
    ax.set_xticks(x)
    ax.set_xticklabels(SUPERCLASSES)
    ax.set_ylabel('Prevalence in split (%)')
    ax.set_xlabel(f'Superclass  |  split by {source}')
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig


def _pick_example_rows(frame: pd.DataFrame, seed: int) -> dict:
    rng = np.random.RandomState(int(seed))
    chosen = {}
    for cls in SUPERCLASSES:
        pool = frame[frame['diagnostic_superclass'].map(lambda xs, c=cls: list(xs) == [c])].copy()
        if pool.empty:
            pool = frame[frame['diagnostic_superclass'].map(lambda xs, c=cls: c in set(xs))].copy()
        ranked = pool
        if 'validated_by_human' in ranked.columns:
            human = ranked[ranked['validated_by_human'] == True]
            if not human.empty:
                ranked = human
        clean = ranked[ranked.apply(_record_is_clean, axis=1)]
        if not clean.empty:
            ranked = clean
        pick = ranked.iloc[int(rng.randint(0, len(ranked)))]
        chosen[cls] = pick
    return chosen


def plot_example_ecgs(data_root: Path, examples: dict, fs: int):
    n_samples = int(round(2.5 * fs))
    fig = plt.figure(figsize=(COL_DOUBLE, 10.4))
    outer = fig.add_gridspec(len(SUPERCLASSES), 1, hspace=0.38)
    for r, cls in enumerate(SUPERCLASSES):
        row = examples[cls]
        signal = np.asarray(load_record_signal(data_root, row, sampling_rate=fs), dtype=np.float32)
        if signal.shape[0] == 12 and signal.shape[1] != 12:
            signal = signal.T
        snippet = signal[:n_samples]
        t = np.arange(snippet.shape[0]) / float(fs)
        inner = outer[r].subgridspec(3, 4, wspace=0.08, hspace=0.12)
        for i in range(3):
            for j in range(4):
                ax = fig.add_subplot(inner[i, j])
                lead = LEAD_GRID[i][j]
                idx = LEAD_NAMES.index(lead)
                ax.plot(t, snippet[:, idx], color='#111111', linewidth=0.45)
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)
                ax.text(0.01, 0.92, lead, transform=ax.transAxes, fontsize=6, va='top')
                if i == 0 and j == 0:
                    ecg_id = int(row['ecg_id'])
                    ax.set_ylabel(f'{cls}  #{ecg_id}', fontsize=8)
    fig.text(0.5, 0.01, '2.5 s excerpt  |  100 Hz  |  12-lead layout', ha='center', fontsize=7)
    return fig


# --- run -----------------------------------------------------------------
apply_paper_style()

if 'DATA_DIR' not in globals() or 'PROJECT_DIR' not in globals():
    raise RuntimeError('Run the path/config cell first (PROJECT_DIR, DATA_DIR).')
if 'find_data_root' not in globals() or 'load_metadata' not in globals():
    raise RuntimeError('Run the helpers cell first.')

if 'DATA_ROOT' not in globals() or 'metadata' not in globals():
    DATA_ROOT = find_data_root(DATA_DIR)
    metadata = load_metadata(DATA_ROOT)
    print('Loaded metadata from', DATA_ROOT)
else:
    print('Using in-session metadata from', DATA_ROOT)

paper_meta = _ensure_ecg_id_column(metadata)
paper_y = _y_matrix(paper_meta)
n_records = int(paper_y.shape[0])
n_patients = int(paper_meta['patient_id'].nunique())
print(f'N records={n_records:,}  patients={n_patients:,}  (empty diagnostic dropped)')

fig01 = plot_superclass_prevalence(paper_meta, paper_y)
save_paper_fig(fig01, 'fig01_superclass_prevalence')

fig02, cooccur_counts, jaccard = plot_cooccurrence(paper_y)
save_paper_fig(fig02, 'fig02_cooccurrence')

fig03, card_counts, top_combos = plot_cardinality_and_combos(paper_meta)
save_paper_fig(fig03, 'fig03_cardinality_and_combos')

fig04 = plot_age_sex(paper_meta)
save_paper_fig(fig04, 'fig04_age_sex')

agg_df = _load_subclass_table(Path(DATA_ROOT))
fig05, subclass_counts, subclass_super = plot_subclass_counts(paper_meta, agg_df)
save_paper_fig(fig05, 'fig05_subclass_counts')

has_session_split = all(name in globals() for name in ('train_df', 'val_df', 'test_df'))
if has_session_split:
    split_frames = {
        'train': _ensure_ecg_id_column(train_df),
        'val': _ensure_ecg_id_column(val_df),
        'test': _ensure_ecg_id_column(test_df),
    }
else:
    split_frames = {
        'train': paper_meta[paper_meta['strat_fold'].isin(TRAIN_FOLDS)],
        'val': paper_meta[paper_meta['strat_fold'].isin(VAL_FOLDS)],
        'test': paper_meta[paper_meta['strat_fold'].isin(TEST_FOLDS)],
    }
split_source = 'strat_fold (1-8 / 9 / 10)'
fig06 = plot_split_balance(split_frames, split_source)
save_paper_fig(fig06, 'fig06_split_balance')

example_seed = int(globals().get('EXAMPLE_SEED', 42))
examples = _pick_example_rows(paper_meta, example_seed)
example_ids = {cls: int(row['ecg_id']) for cls, row in examples.items()}
fs = int(globals().get('SAMPLING_RATE', 100))
fig07 = plot_example_ecgs(Path(DATA_ROOT), examples, fs)
save_paper_fig(fig07, 'fig07_example_ecgs')

prevalence = _prevalence_counts(paper_y)
stats = {
    'n_records': n_records,
    'n_patients': n_patients,
    'sampling_rate_hz': fs,
    'classes': list(SUPERCLASSES),
    'prevalence': prevalence,
    'cardinality': {str(k): int(v) for k, v in card_counts.items()},
    'top_combinations': [{'labels': k, 'n': int(v)} for k, v in top_combos],
    'cooccurrence_counts': {
        SUPERCLASSES[i]: {SUPERCLASSES[j]: int(cooccur_counts[i, j]) for j in range(5)}
        for i in range(5)
    },
    'jaccard': {
        SUPERCLASSES[i]: {SUPERCLASSES[j]: float(jaccard[i, j]) for j in range(5)}
        for i in range(5)
    },
    'subclass_counts': {k: int(v) for k, v in subclass_counts.items()},
    'split_source': split_source,
    'split_sizes': {k: int(len(v)) for k, v in split_frames.items()},
    'example_ecg_ids': example_ids,
    'example_seed': example_seed,
}
(PAPER_FIG_DIR / 'dataset_stats.json').write_text(json.dumps(stats, indent=2), encoding='utf-8')

rows = []
for name, item in prevalence.items():
    rows.append({'metric': f'prevalence_{name}', 'n': item['n'], 'pct': item['pct']})
for k, v in stats['cardinality'].items():
    rows.append({'metric': f'cardinality_{k}', 'n': v, 'pct': 100.0 * v / n_records})
pd.DataFrame(rows).to_csv(PAPER_FIG_DIR / 'dataset_stats.csv', index=False)
print('stats ->', PAPER_FIG_DIR / 'dataset_stats.json')

print('\nSelected example ECG IDs (seed=%s):' % example_seed)
for cls in SUPERCLASSES:
    print(f'  {cls}: {example_ids[cls]}')

captions = {
    'fig01_superclass_prevalence': (
        'Figure. Prevalence of the five PTB-XL diagnostic superclasses. '
        'Counts are multi-label (a record may contribute to more than one class). '
        f'N = {n_records:,} records from {n_patients:,} patients after excluding ECGs without a diagnostic superclass.'
    ),
    'fig02_cooccurrence': (
        'Figure. Pairwise co-occurrence of diagnostic superclasses. Each cell shows the number of records '
        'labelled with both classes (top) and the Jaccard index (bottom). The diagonal is class prevalence.'
    ),
    'fig03_cardinality_and_combos': (
        'Figure. Multi-label structure of PTB-XL superclasses. Left: number of diagnostic superclasses per record. '
        'Right: the ten most frequent label combinations (classes ordered as NORM, MI, STTC, CD, HYP).'
    ),
    'fig04_age_sex': (
        'Figure. Age distribution by sex for diagnostic-labelled PTB-XL records '
        '(male = 0, female = 1 in the source metadata). Bars use 5-year bins; ages missing in the table are omitted.'
    ),
    'fig05_subclass_counts': (
        'Figure. Diagnostic subclass counts (SCP-ECG), coloured by parent superclass. '
        'A record may carry multiple subclasses. Mapping follows scp_statements.csv (diagnostic = 1).'
    ),
    'fig06_split_balance': (
        f'Figure. Superclass prevalence within each split ({split_source}). '
        'Percentages are multi-label fractions of records in that split.'
    ),
    'fig07_example_ecgs': (
        'Figure. Representative 12-lead excerpts (2.5 s, 100 Hz) for single-label records of each superclass. '
        'Leads follow the standard 3 x 4 layout. Record IDs: '
        + ', '.join(f'{k}={v}' for k, v in example_ids.items())
        + '. Selection prefers human-validated traces without annotated noise (seed '
        + str(example_seed)
        + ').'
    ),
}

display(Markdown('### Suggested captions'))
for stem, text in captions.items():
    print(f'\n[{stem}]\n{text}')

print('\nAll figures:', PAPER_FIG_DIR)
